# 🔧 utils.py — Shared Utilities
Data loading, preprocessing, scaling, sequence generation and evaluation metrics used by all other notebooks.

## Imports & Constants

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH  = "data/UPI_Master_2021_2025.csv"
TRAIN_END  = "2025-09-30"
TEST_START = "2025-10-01"


## load_data()

In [ ]:
def load_data(path=DATA_PATH):
    df = pd.read_csv(path, parse_dates=["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    df["Festival_Name"] = df["Festival_Name"].fillna("")
    df["Holiday_Type"]  = df["Holiday_Type"].fillna("Unknown")
    df["Month_Num"]     = df["Date"].dt.month
    df["Day_Of_Year"]   = df["Date"].dt.dayofyear
    df["Week_Of_Year"]  = df["Date"].dt.isocalendar().week.astype(int)
    df["Quarter"]       = df["Date"].dt.quarter
    return df


## train_test_split()

In [ ]:
def train_test_split(df):
    train = df[df["Date"] <= TRAIN_END].copy()
    test  = df[df["Date"] >= TEST_START].copy()
    return train, test


## Helper utilities

In [ ]:
def get_monthly(df):
    return (
        df.groupby(pd.Grouper(key="Date", freq="MS"))
          .agg({"Volume (In Mn.)": "sum", "Value (In Cr.)": "sum"})
          .reset_index()
    )

def get_exog_cols():
    return ["Is_Weekend", "Is_Festival", "Is_Holiday_Adjacent",
            "Is_Long_Weekend", "Holiday_Cluster_7D"]

def scale_series(train_vals, test_vals):
    mn, mx = train_vals.min(), train_vals.max()
    return (train_vals-mn)/(mx-mn), (test_vals-mn)/(mx-mn), mn, mx

def inverse_scale(vals, mn, mx):
    return vals * (mx - mn) + mn

def make_sequences(data, window=30):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i])
        y.append(data[i])
    return np.array(X), np.array(y)

def evaluation_metrics(actual, predicted):
    actual, predicted = np.array(actual), np.array(predicted)
    mae  = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted)**2))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    return {"MAE": round(mae,4), "RMSE": round(rmse,4), "MAPE": round(mape,4)}


## Quick sanity check

In [ ]:
df = load_data()
train, test = train_test_split(df)
print(f"Total rows  : {len(df):,}")
print(f"Date range  : {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Train rows  : {len(train):,}  ({train['Date'].min().date()} → {train['Date'].max().date()})")
print(f"Test rows   : {len(test):,}   ({test['Date'].min().date()} → {test['Date'].max().date()})")
df.head(3)
